# ChatGPT Dataset: Cleaning & Analysis

This notebook cleans the synthetic Ferritic/Martensitic steel dataset generated by
ChatGPT, then checks it against known metallurgical trends.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

sns.set_theme(style="whitegrid", context="paper", font="sans-serif")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'sans-serif'
})


## 1. Cleaning the data

ChatGPT's output needed the same kind of fixes as Grok's: phase fractions embedded in
text, mixed hardness scales, and a few unit/negative-value errors.


In [ ]:
def clean_chatgpt_dataset(filepath):
    """
    Cleans the ChatGPT dataset.

    - Standardizes column names to match the shared schema
    - Extracts numeric ferrite/martensite fractions from text
    - Converts HRC hardness values to HV
    - Fixes unit errors in elastic modulus and removes negative values
    - Imputes remaining gaps with the column median (numeric) or mode (categorical)
    """
    if not os.path.exists(filepath):
        print(f"Error: File '{filepath}' not found.")
        return None

    df = pd.read_csv(filepath)
    print(f"Processing '{filepath}' | Initial shape: {df.shape}")

    rename_map = {
        'Steel_Class': 'Alloy_Type',
        'Sample_ID': 'ID',
        'Yield_Strength_MPa': 'YS_MPa',
        'Ultimate_Tensile_Strength_MPa': 'UTS_MPa',
        'Elastic_Modulus_GPa': 'Modulus_GPa',
        'Elongation_pct': 'Elongation_%',
        'Hardness': 'Hardness_Value',
        'Hardness_Scale': 'Hardness_Scale',
        'Anneal_Temp_C': 'Heat_Treat_Temp_C',
        'Anneal_Time_h': 'Heat_Treat_Time_h'
    }
    df.rename(columns=rename_map, inplace=True)

    # phase fractions come as text, e.g. "Ferrite:95;Carbides:5" -> pull out the numbers
    if 'Phases_VolFrac' in df.columns:
        def extract_phase(text, phase_name):
            pattern = rf"{phase_name}[^0-9]*(\d+(\.\d+)?)"
            match = re.search(pattern, str(text), re.IGNORECASE)
            return float(match.group(1)) if match else 0.0

        df['Ferrite_Vol_%'] = df['Phases_VolFrac'].apply(lambda x: extract_phase(x, 'Ferrite'))
        df['Martensite_Vol_%'] = df['Phases_VolFrac'].apply(lambda x: extract_phase(x, 'Martensite'))
        df.drop(columns=['Phases_VolFrac'], inplace=True)

    # hardness: convert HRC to HV so both scales are comparable
    if 'Hardness_Value' in df.columns and 'Hardness_Scale' in df.columns:
        df['Hardness_Scale'] = df['Hardness_Scale'].astype(str).str.upper().str.strip()

        def normalize_hardness(row):
            val = row['Hardness_Value']
            scale = row['Hardness_Scale']
            if pd.isna(val):
                return np.nan
            if scale == 'HV':
                return val
            elif scale == 'HRC':
                return (9.0 * val) + 55.0
            return val

        df['Hardness_HV'] = df.apply(normalize_hardness, axis=1)
        df.drop(columns=['Hardness_Value', 'Hardness_Scale'], inplace=True)
        print("  - Standardized hardness to a single 'Hardness_HV' column.")

    # unit fix: modulus values over 1000 are in MPa, not GPa
    if 'Modulus_GPa' in df.columns:
        mask_units = df['Modulus_GPa'] > 1000
        if mask_units.any():
            print(f"  - Fixed {mask_units.sum()} rows in Modulus (MPa -> GPa).")
            df.loc[mask_units, 'Modulus_GPa'] = df.loc[mask_units, 'Modulus_GPa'] / 1000.0

    # remove physically impossible negative values
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    cols_to_check = [c for c in numeric_cols if 'Temp' not in c]
    for col in cols_to_check:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  - Removed {neg_count} negative values in {col}.")
            df.loc[df[col] < 0, col] = np.nan

    # fill remaining gaps
    for col in numeric_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])

    df['Source'] = 'ChatGPT'
    return df


filename = '../data/raw/chatgpt.csv'
if os.path.exists(filename):
    chatgpt_clean = clean_chatgpt_dataset(filename)
    if chatgpt_clean is not None:
        chatgpt_clean.to_csv('../data/cleaned/cleaned_chatgpt.csv', index=False)
        print("\nSaved cleaned data to 'cleaned_ChatGPT.csv'.")
        print(chatgpt_clean[['Alloy_Type', 'Ferrite_Vol_%', 'Martensite_Vol_%']].head(3))
        print(chatgpt_clean['Hardness_HV'].describe().round(1))


## 2. Summary statistics


In [ ]:
filename = '../data/cleaned/cleaned_chatgpt.csv'
if os.path.exists(filename):
    df = pd.read_csv(filename)

    target_cols = [
        'YS_MPa', 'UTS_MPa', 'Elongation_%',
        'Hardness_HV', 'Ferrite_Vol_%', 'Modulus_GPa'
    ]
    existing_cols = [c for c in target_cols if c in df.columns]
    stats_df = df[existing_cols]

    summary_stats = stats_df.describe().T
    summary_stats['Median'] = stats_df.median()
    summary_stats['Mode'] = stats_df.mode().iloc[0]
    summary_stats['Skewness'] = stats_df.skew()

    final_stats = summary_stats[['mean', 'Median', 'Mode', 'std', 'min', 'max', 'Skewness']]
    print(final_stats.round(2))

    final_stats.round(2).to_csv('../data/cleaned/chatgpt_statistics_summary.csv')
else:
    print(f"'{filename}' not found. Run the cleaning cell first.")


## 3. Exploratory plots


In [ ]:
filename = '../data/cleaned/cleaned_chatgpt.csv'

if os.path.exists(filename):
    df = pd.read_csv(filename)

    sns.set_theme(style="whitegrid", context="paper", font="sans-serif", font_scale=1.2)
    custom_palette = {"Ferritic": "#2b8cbe", "Martensitic": "#d73027"}
    unique_classes = df['Alloy_Type'].unique()
    for cls in unique_classes:
        if cls not in custom_palette:
            custom_palette[cls] = "#95a5a6"

    numeric_df = df.select_dtypes(include=[np.number])
    if 'ID' in numeric_df.columns:
        numeric_df = numeric_df.drop(columns=['ID'])

    plt.figure(figsize=(14, 12))
    sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap="coolwarm",
                center=0, square=True, linewidths=.5, cbar_kws={"shrink": .8})
    plt.title('Correlation Matrix of All Variables', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

    if 'Grain_Size_um' in df.columns:
        plt.figure(figsize=(9, 7))
        sns.scatterplot(data=df, x='Grain_Size_um', y='YS_MPa', hue='Alloy_Type',
                        palette=custom_palette, s=120, edgecolor='black', alpha=0.8)
        sns.regplot(data=df, x='Grain_Size_um', y='YS_MPa', scatter=False,
                    order=2, color='#34495e', line_kws={'linestyle':'--'})
        plt.title('Yield Strength vs. Grain Size (Hall-Petch)', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Grain Size (\u00b5m)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.legend(title='Alloy Type')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.scatterplot(data=df, x='YS_MPa', y='Elongation_%', hue='Alloy_Type',
                    palette=custom_palette, style='Alloy_Type', s=120, alpha=0.9)
    plt.title('Yield Strength vs. Elongation', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
    plt.ylabel('Elongation (%)', fontsize=14, fontweight='bold')
    plt.legend(title='Alloy Type')
    plt.tight_layout()
    plt.show()

    if 'C_wt%' in df.columns:
        plt.figure(figsize=(9, 7))
        sns.scatterplot(data=df, x='C_wt%', y='Hardness_HV', hue='Alloy_Type',
                        palette=custom_palette, s=120, edgecolor='black')
        plt.title('Hardness vs. Carbon Content', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Carbon Content (wt%)', fontsize=14, fontweight='bold')
        plt.ylabel('Hardness (HV)', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.regplot(data=df, x='Hardness_HV', y='UTS_MPa',
                scatter_kws={'s': 80, 'alpha': 0.6, 'color':'purple'},
                line_kws={'color':'black', 'linestyle':'--'})
    plt.title('UTS vs. Hardness', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Hardness (HV)', fontsize=14, fontweight='bold')
    plt.ylabel('Ultimate Tensile Strength (MPa)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    if 'Ferrite_Vol_%' in df.columns and 'C_wt%' in df.columns:
        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=df, x='Ferrite_Vol_%', y='YS_MPa', hue='Alloy_Type',
                        size='C_wt%', sizes=(50, 300), palette=custom_palette,
                        alpha=0.7, edgecolor='black')
        plt.title('Yield Strength vs. Ferrite Fraction', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Ferrite Volume Fraction (%)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.gca().invert_xaxis()
        plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title="Alloy & Carbon %")
        plt.tight_layout()
        plt.show()

    if 'Heat_Treat_Temp_C' in df.columns:
        plt.figure(figsize=(9, 7))
        df['Process_Label'] = df['Alloy_Type'].map({
            'Ferritic': 'Ferritic (Annealed)',
            'Martensitic': 'Martensitic (Quenched)'
        }).fillna(df['Alloy_Type'])

        sns.scatterplot(data=df, x='Heat_Treat_Temp_C', y='YS_MPa', hue='Process_Label',
                        palette={'Ferritic (Annealed)': '#2b8cbe', 'Martensitic (Quenched)': '#d73027'},
                        s=100, alpha=0.7, legend=False)
        sns.lineplot(data=df, x='Heat_Treat_Temp_C', y='YS_MPa', hue='Process_Label',
                     palette={'Ferritic (Annealed)': '#2b8cbe', 'Martensitic (Quenched)': '#d73027'},
                     marker=None, linewidth=3, errorbar=None)

        plt.title('Yield Strength vs. Heat Treatment Temperature', fontsize=16, fontweight='bold', pad=20)
        plt.xlabel('Heat Treatment Temperature (\u00b0C)', fontsize=14, fontweight='bold')
        plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
        plt.legend(title='Alloy Condition')
        plt.tight_layout()
        plt.show()

    plt.figure(figsize=(9, 7))
    sns.violinplot(data=df, x='Alloy_Type', y='YS_MPa', hue='Alloy_Type',
                   palette=custom_palette, inner="quartile", dodge=False)
    plt.title('Yield Strength Distribution by Alloy', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Alloy Type', fontsize=14, fontweight='bold')
    plt.ylabel('Yield Strength (MPa)', fontsize=14, fontweight='bold')
    plt.legend([],[], frameon=False)
    plt.tight_layout()
    plt.show()

else:
    print(f"'{filename}' not found.")


## Observations

ChatGPT's data was numerically well-behaved (values stayed within realistic ranges) but
lacked clear physical trends in several plots. Yield strength vs. elongation formed a
"box-like" cluster rather than a clear trade-off curve, hardness vs. carbon content
looked more like a step function than a gradient (as if carbon were treated as a
category rather than a continuous variable), and the ferrite/martensite phase fraction
data clustered almost entirely at 0% or 100%, without the intermediate dual-phase points
a real dataset would show.
